In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e4/sample_submission.csv
/kaggle/input/playground-series-s5e4/train.csv
/kaggle/input/playground-series-s5e4/test.csv


In [2]:
from tqdm import tqdm
from itertools import combinations

In [3]:
train = pd.read_csv('/kaggle/input/playground-series-s5e4/train.csv', index_col = 'id')
test = pd.read_csv('/kaggle/input/playground-series-s5e4/test.csv', index_col = 'id')
sub = pd.read_csv('/kaggle/input/playground-series-s5e4/sample_submission.csv')

In [4]:
train.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
id,,,,,,,,,,,
0,Mystery Matters,Episode 98,NaN,True Crime,74.81,Thursday,Night,NaN,0.0,Positive,31.41998
1,Joke Junction,Episode 26,119.80,Comedy,66.95,Saturday,Afternoon,75.95,2.0,Negative,88.01241
2,Study Sessions,Episode 16,73.90,Education,69.97,Tuesday,Evening,8.97,0.0,Negative,44.92531
3,Digital Digest,Episode 45,67.17,Technology,57.22,Monday,Morning,78.70,2.0,Positive,46.27824
4,Mind & Body,Episode 86,110.51,Health,80.07,Monday,Afternoon,58.68,3.0,Neutral,75.61031


In [5]:
target = train['Listening_Time_minutes']
target

id
0          31.41998
1          88.01241
2          44.92531
3          46.27824
4          75.61031
            ...    
749995     56.87058
749996     45.46242
749997     15.26000
749998    100.72939
749999     11.94439
Name: Listening_Time_minutes, Length: 750000, dtype: float64

In [6]:
# train['Episode_Title'] をEpisoce と　数値に分けます

train[['Episode', 'Episode_Number']] = train['Episode_Title'].str.split('Episode ', expand=True)
train['Episode_Number'] = train['Episode_Number'].astype(float)
train.drop(['Episode_Title','Episode'], axis=1, inplace=True)

test[['Episode', 'Episode_Number']] = test['Episode_Title'].str.split('Episode ', expand=True)
test['Episode_Number'] = test['Episode_Number'].astype(float)
test.drop(['Episode_Title','Episode'], axis=1, inplace=True)


In [7]:
target_col = 'Listening_Time_minutes'
target = train[target_col]
train.drop(target_col, axis=1, inplace=True)

In [8]:
train.isnull().sum()

Podcast_Name                        0
Episode_Length_minutes          87093
Genre                               0
Host_Popularity_percentage          0
Publication_Day                     0
Publication_Time                    0
Guest_Popularity_percentage    146030
Number_of_Ads                       1
Episode_Sentiment                   0
Episode_Number                      0
dtype: int64

In [9]:
for col in train.columns:
    print(col, train[col].unique())
    train[col].fillna(train[col].mode()[0], inplace=True)
    test[col].fillna(train[col].mode()[0],inplace = True)

train.describe()

Podcast_Name ['Mystery Matters' 'Joke Junction' 'Study Sessions' 'Digital Digest'
 'Mind & Body' 'Fitness First' 'Criminal Minds' 'News Roundup'
 'Daily Digest' 'Music Matters' 'Sports Central' 'Melody Mix' 'Game Day'
 'Gadget Geek' 'Global News' 'Tech Talks' 'Sport Spot' 'Funny Folks'
 'Sports Weekly' 'Business Briefs' 'Tech Trends' 'Innovators'
 'Health Hour' 'Comedy Corner' 'Sound Waves' 'Brain Boost'
 "Athlete's Arena" 'Wellness Wave' 'Style Guide' 'World Watch' 'Humor Hub'
 'Money Matters' 'Healthy Living' 'Home & Living' 'Educational Nuggets'
 'Market Masters' 'Learning Lab' 'Lifestyle Lounge' 'Crime Chronicles'
 'Detective Diaries' 'Life Lessons' 'Current Affairs' 'Finance Focus'
 'Laugh Line' 'True Crime Stories' 'Business Insights' 'Fashion Forward'
 'Tune Time']
Episode_Length_minutes [         nan 119.8         73.9        ... 112.002        6.71292308
  62.16729385]


/tmp/ipykernel_13/2494551127.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna(train[col].mode()[0], inplace=True)
/tmp/ipykernel_13/2494551127.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try

Genre ['True Crime' 'Comedy' 'Education' 'Technology' 'Health' 'News' 'Music'
 'Sports' 'Business' 'Lifestyle']
Host_Popularity_percentage [74.81    66.95    69.97    ...  1.47    30.116   86.64906]
Publication_Day ['Thursday' 'Saturday' 'Tuesday' 'Monday' 'Sunday' 'Wednesday' 'Friday']
Publication_Time ['Night' 'Afternoon' 'Evening' 'Morning']
Guest_Popularity_percentage [    nan  75.95    8.97  ...  99.597 107.58  105.44 ]
Number_of_Ads [  0.     2.     3.     1.    53.37    nan 103.91 103.    53.42 103.75
  12.   103.25 103.88]
Episode_Sentiment ['Positive' 'Negative' 'Neutral']
Episode_Number [ 98.  26.  16.  45.  86.  19.  47.  44.  32.  81.  66.  62.  76.  37.
  20.  82.  72.  61. 100.  54.  17.  36.  97.  27.  31.  88.  38.  92.
  74.  30.  63.  67.  77.   4.  93.  24.   1.   2.  25.  56.  75.  12.
  21.   6.  85.  23.  33.   7.  53.  15.  43.  71.  69.  13.  89.   3.
  64.  73.  79.  94.  80.  42.  10.  48.  96.  40.  49.   9.  50.  78.
  84.  87.  58.  57.  59.  39.  46.  91. 

,Episode_Length_minutes,Host_Popularity_percentage,Guest_Popularity_percentage,Number_of_Ads,Episode_Number
count,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000
mean,57.780609,59.859901,55.408912,1.348853,51.445811
std,36.123592,22.873098,26.334186,1.151131,28.085623
min,0.000000,1.300000,0.000000,0.000000,1.000000
25%,26.400000,39.410000,34.550000,0.000000,28.000000
50%,56.610000,60.050000,64.650000,1.000000,52.000000
75%,90.310000,79.530000,71.040000,2.000000,75.000000
max,325.240000,119.460000,119.910000,103.910000,100.000000


In [10]:
train.isnull().sum()

Podcast_Name                   0
Episode_Length_minutes         0
Genre                          0
Host_Popularity_percentage     0
Publication_Day                0
Publication_Time               0
Guest_Popularity_percentage    0
Number_of_Ads                  0
Episode_Sentiment              0
Episode_Number                 0
dtype: int64

In [11]:
def round_num(df):
    df['Episode_Length_minutes'] = df['Episode_Length_minutes'].round().astype('int')
    df['Guest_Popularity_percentage'] = df['Guest_Popularity_percentage'].round().astype('int')
    df['Host_Popularity_percentage'] = df['Host_Popularity_percentage'].round().astype('int')
    df['Number_of_Ads'] = df['Number_of_Ads'].round().astype('int')
    df['Episode_Number'] = df['Episode_Number'].round().astype('int')
    return df


In [12]:
train_df = round_num(train)
test_df = round_num(test)

In [13]:
cat_cols = ['Podcast_Name','Genre', 'Publication_Day', 'Publication_Time', 'Episode_Sentiment']

In [14]:
from sklearn.preprocessing import LabelEncoder

for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

In [15]:
train_df

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Episode_Number
id,,,,,,,,,,
0,34,7,9,75,4,3,69,0,2,98
1,24,120,1,67,2,0,76,2,0,26
2,40,74,2,70,5,1,9,0,0,16
3,10,67,8,57,1,2,79,2,2,45
4,31,111,3,80,1,0,59,3,1,86
...,...,...,...,...,...,...,...,...,...,...
749995,26,76,2,69,2,2,69,0,0,25
749996,2,76,0,35,2,3,69,2,1,21
749997,28,31,4,79,4,2,85,0,0,51


In [16]:
from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_squared_error
X_train, X_test, y_train, y_test = train_test_split(train_df, target, test_size=0.2, random_state=42)

In [17]:
from bayes_opt import BayesianOptimization
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Define the objective function for Bayesian Optimization
def rf_cv(n_estimators, max_depth, min_samples_split, min_samples_leaf):
    val = int(n_estimators)
    max_depth = int(max_depth)
    min_samples_split = int(min_samples_split)
    min_samples_leaf = int(min_samples_leaf)

    model = RandomForestRegressor(n_estimators=val,
                                  max_depth=max_depth,
                                  min_samples_split=min_samples_split,
                                  min_samples_leaf=min_samples_leaf,
                                  random_state=42,
                                  criterion='squared_error',
                                  n_jobs=-1)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test,y_pred))
    return -rmse  # BayesianOptimization maximizes


# Define the parameter bounds for Bayesian Optimization
pbounds = {
    'n_estimators': (100, 1000),
    'max_depth': (5, 30),
    'min_samples_split': (2, 20),
    'min_samples_leaf': (1, 10),
}

# Initialize the Bayesian Optimization object
optimizer = BayesianOptimization(f=rf_cv, pbounds=pbounds, random_state=42)


# Perform Bayesian Optimization
optimizer.maximize(init_points=15, n_iter=45)

# Get the best parameters and RMSE
print(optimizer.max)

# Train the model with the best parameters
best_params = optimizer.max['params']
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['max_depth'] = int(best_params['max_depth'])
best_params['min_samples_split'] = int(best_params['min_samples_split'])
best_params['min_samples_leaf'] = int(best_params['min_samples_leaf'])

print('best_params: ', best_params)


best_model = RandomForestRegressor(**best_params, random_state=42, criterion='squared_error')
best_model.fit(X_train, y_train)


# Make predictions on the test set
y_pred_best = best_model.predict(test_df)

# Create submission file
sub['Listening_Time_minutes'] = y_pred_best
sub.to_csv('/kaggle/working/submission.csv', index=False)
sub

|   iter    |  target   | max_depth | min_sa... | min_sa... | n_esti... |
-------------------------------------------------------------------------
| 1         | -13.38    | 14.36     | 9.556     | 15.18     | 638.8     |
| 2         | -13.6     | 8.9       | 2.404     | 3.046     | 879.6     |
| 3         | -13.22    | 20.03     | 7.373     | 2.371     | 972.9     |
| 4         | -13.13    | 25.81     | 2.911     | 5.273     | 265.1     |
| 5         | -13.46    | 12.61     | 5.723     | 9.775     | 362.1     |
| 6         | -13.17    | 20.3      | 2.255     | 7.259     | 429.7     |
| 7         | -13.31    | 16.4      | 8.067     | 5.594     | 562.8     |
| 8         | -13.22    | 19.81     | 1.418     | 12.94     | 253.5     |
| 9         | -13.69    | 6.626     | 9.54      | 19.38     | 827.6     |
| 10        | -13.46    | 12.62     | 1.879     | 14.32     | 496.1     |
| 11        | -13.61    | 8.051     | 5.457     | 2.619     | 918.4     |
| 12        | -13.5     | 11.47     | 

,id,Listening_Time_minutes
0,750000,56.153452
1,750001,21.329495
2,750002,49.145228
3,750003,77.624617
4,750004,47.081497
...,...,...
249995,999995,11.155657
249996,999996,60.564195
249997,999997,7.837605
249998,999998,72.315208


In [18]:
best_model

RandomForestRegressor(max_depth=29, min_samples_leaf=3, n_estimators=540,
                      random_state=42)